# RSNA Knee - v7 A: build the 384 px image store (CPU only, no GPU needed, Internet OFF is fine)

Same steps as our v6 store (the same functions: series choice, slice order, left knees mirrored, 1-99 % window,
JPEG), only sharper: **384 px for 160 mm** instead of 288 px for 140 mm. The training notebook then reads the
140 mm view at **336 px** with no upscaling, and its random 125-160 mm training views have real pixels.

**Run it 4 times** (4 sessions, in parallel if you like): `PART = 0`, `1`, `2`, `3`. Each run makes one part.
After each run: **Output -> New Dataset** named `knee-store-v7-<PART>`. Attach all 4 parts to the training notebook
(and **not** the old v6 parts in the same session: the two stores have different settings).

The last cell checks the part: every study is there, every slice decodes, and the pictures match a fresh read.


In [1]:
PART = 3                      # 0, 1, 2 or 3: run this notebook once per part
N_PARTS = 4
IMG_SIZE = 384                # pixels per side of every stored slice
CROP_MM = 160.0               # real size of the stored square (the model reads the centre 140 mm = 336 px)
SPAN = [0.06, 0.94]           # part of every series that is kept (the outer 6 % are mostly empty)
MAX_SLICES = 64               # very long series: an even subset of 64 slices
JPEG_QUALITY = 95
WORKERS = 4                   # Kaggle CPU sessions have 4 cores
MAX_PART_GB = 18.0            # Kaggle keeps at most about 19.5 GB of output: stop cleanly before that


In [2]:
import os, sys, glob, json, time, subprocess, importlib

OUT = "/kaggle/working"
T0 = time.time()


def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)


# The competition DICOMs come in 4 transfer syntaxes. Plain pydicom cannot read "JPEG Lossless" (needs pylibjpeg or
# gdcm), and a slice it cannot read silently becomes black. So: install the decoders (Internet ON) and keep their wheel
# files in the output, so the offline submission notebook can install the very same ones.
DECODERS = ["pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "python-gdcm"]
WHEELS = f"{OUT}/dicom_wheels"
if PART == 0:
    r = subprocess.run([sys.executable, "-m", "pip", "download", "--no-deps", "-q", "-d", WHEELS] + DECODERS,
                       capture_output=True, text=True)
    log("decoder wheels saved for the submission notebook" if r.returncode == 0 else
        f"warning: could not download decoder wheels (Internet OFF?): {r.stderr[-300:]}")
local = sorted(glob.glob(f"{WHEELS}/*.whl")) + sorted(q for d in range(1, 5) for q in glob.glob("/kaggle/input/" + "*/" * d + "dicom_wheels/*.whl"))
cmd = [sys.executable, "-m", "pip", "install", "--no-deps", "-q"] + (["--no-index"] + local if local else DECODERS)
r = subprocess.run(cmd, capture_output=True, text=True)
log("DICOM decoders installed" if r.returncode == 0 else f"warning: decoder install failed: {r.stderr[-300:]}")
importlib.invalidate_caches()

import numpy as np
import pandas as pd
import cv2
import pydicom
from multiprocessing import Pool

try:
    from pydicom.pixels import apply_voi_lut
except ImportError:
    from pydicom.pixel_data_handlers.util import apply_voi_lut

COMP = next(p for p in ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
                        "/kaggle/input/rsna-knee-abnormality-detection"] if os.path.exists(p))
PLANES = ["Sagittal", "Coronal", "Axial"]
log(f"data root: {COMP} | pydicom {pydicom.__version__}")


[   0.2 min] DICOM decoders installed
[   0.2 min] data root: /kaggle/input/competitions/rsna-knee-abnormality-detection | pydicom 3.0.2


In [3]:
def plane_blocks(prep):
    """Where each plane's slices sit inside one study's array."""
    blocks, start = {}, 0
    for plane in PLANES:
        n = prep["slices"][plane]
        blocks[plane] = (start, start + n)
        start += n
    return blocks


def pick_series(series_df):
    """One series per plane. Fluid-sensitive first (in this data it always equals fat-suppressed)."""
    series_df = series_df.copy()
    for col, empty in (("Fluid_Sensitive", 0), ("Fat_Suppression", 0), ("Anatomical_Plane", ""), ("SeriesInstanceUID", "")):
        if col not in series_df.columns:
            series_df[col] = empty                    # a missing column must not crash the hidden test run
    if "StudyInstanceUID" not in series_df.columns or series_df.empty:
        return pd.DataFrame(columns=["StudyInstanceUID"] + PLANES)
    ranked = series_df.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
    rows = []
    for study, group in ranked.groupby("StudyInstanceUID"):
        row = {"StudyInstanceUID": str(study)}
        for plane in PLANES:
            same = group[group["Anatomical_Plane"] == plane]
            row[plane] = str(same.iloc[0]["SeriesInstanceUID"]) if len(same) else ""
        rows.append(row)
    return pd.DataFrame(rows)


def slice_key(header):
    """Position along the body axis the stack runs along (x for sagittal, y for coronal, z for axial).
    Using the body axis, not the scanner's slice direction, gives the same order on every scanner."""
    try:
        pos = np.array(header.ImagePositionPatient, dtype=float)
        orient = np.array(header.ImageOrientationPatient, dtype=float)
        normal = np.cross(orient[:3], orient[3:])
        return float(pos[int(np.argmax(np.abs(normal)))])
    except Exception:
        return float(getattr(header, "InstanceNumber", 0) or 0)


def sorted_paths(folder):
    ordered = []
    for path in glob.glob(os.path.join(folder, "*.dcm")):
        try:
            header = pydicom.dcmread(path, stop_before_pixels=True,
                                     specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"])
            ordered.append((slice_key(header), path))
        except Exception:
            continue
    ordered.sort(key=lambda item: item[0])
    return [path for _, path in ordered]


def read_image(path):
    """One slice as floats, plus mm per pixel along rows and columns."""
    try:
        data = pydicom.dcmread(path)
        try:
            image = apply_voi_lut(data.pixel_array, data).astype(np.float32)
        except Exception:
            image = np.asarray(data.pixel_array, dtype=np.float32)
        image = np.nan_to_num(np.squeeze(image), nan=0.0, posinf=0.0, neginf=0.0)
        if image.ndim != 2:
            return None, None
        if getattr(data, "PhotometricInterpretation", "") == "MONOCHROME1":
            image = image.max() - image
        spacing = getattr(data, "PixelSpacing", None)
        mm = (float(spacing[0]), float(spacing[1])) if spacing is not None and len(spacing) >= 2 else None
        return image, mm
    except Exception:
        return None, None


def crop_to_mm(image, mm, crop_mm):
    """Keep the same real area of knee, whatever the scanner's pixel size."""
    if not mm or mm[0] <= 0 or mm[1] <= 0:
        return image
    half_y, half_x = int(round(crop_mm / mm[0] / 2)), int(round(crop_mm / mm[1] / 2))
    mid_y, mid_x = image.shape[0] // 2, image.shape[1] // 2
    cropped = image[max(mid_y - half_y, 0):mid_y + half_y, max(mid_x - half_x, 0):mid_x + half_x]
    return cropped if cropped.size else image


def load_series(folder, n, prep):
    size = prep["img_size"]
    empty = np.zeros((n, size, size), dtype=np.uint8)
    paths = sorted_paths(folder)
    if not paths:
        return empty

    start = int(prep["span"][0] * len(paths))
    stop = max(int(prep["span"][1] * len(paths)), start + 1)
    picked = np.linspace(start, stop - 1, n).round().astype(int)

    slices, last_good = [], None
    for i in picked:
        image, mm = read_image(paths[i])
        if image is None:
            slices.append(last_good if last_good is not None else np.zeros((size, size), np.float32))
            continue
        image = cv2.resize(crop_to_mm(image, mm, prep["crop_mm"]), (size, size), interpolation=cv2.INTER_AREA)
        slices.append(image)
        last_good = image

    stack = np.stack(slices).astype(np.float32)
    low, high = np.percentile(stack, [1, 99])         # one window for the whole series
    stack = (np.clip(stack, low, high) - low) / max(high - low, 1e-6)
    return (stack * 255).astype(np.uint8)


def knee_side(study_uid, row, root):
    """'L', 'R' or None. Laterality tag first, else where the image centre sits (right knee at x < 0)."""
    tags, centres = [], []
    for plane in PLANES:
        series_uid = row.get(plane, "")
        if not isinstance(series_uid, str) or not series_uid:
            continue
        files = glob.glob(os.path.join(root, study_uid, series_uid, "*.dcm"))
        if not files:
            continue
        try:
            h = pydicom.dcmread(files[0], stop_before_pixels=True)
        except Exception:
            continue
        tag = str(getattr(h, "Laterality", "") or "").strip().upper()
        if tag[:1] in ("L", "R"):
            tags.append(tag[0])
        try:
            pos = np.array(h.ImagePositionPatient, dtype=float)
            orient = np.array(h.ImageOrientationPatient, dtype=float)
            spacing = np.array(h.PixelSpacing, dtype=float)
            centre = (pos + orient[:3] * spacing[1] * float(h.Columns) / 2
                          + orient[3:] * spacing[0] * float(h.Rows) / 2)
            centres.append(centre[0])
        except Exception:
            pass
    if tags:
        return tags[0]
    if centres and abs(float(np.median(centres))) >= 20:
        return "R" if float(np.median(centres)) < 0 else "L"
    return None


def load_study(study_uid, row, root, prep):
    """One study as (all slices, size, size) uint8: sagittal block, coronal block, axial block.
    Left knees are mirrored so every knee looks like a right knee. Missing planes stay black."""
    blocks = plane_blocks(prep)
    total = blocks[PLANES[-1]][1]
    volume = np.zeros((total, prep["img_size"], prep["img_size"]), dtype=np.uint8)

    for plane in PLANES:
        series_uid = row.get(plane, "")
        if isinstance(series_uid, str) and series_uid:
            a, b = blocks[plane]
            try:
                volume[a:b] = load_series(os.path.join(root, study_uid, series_uid), b - a, prep)
            except Exception:
                pass

    try:
        if knee_side(study_uid, row, root) == "L":
            a, b = blocks["Sagittal"]
            volume[a:b] = volume[a:b][::-1]                  # sagittal: read the stack backwards
            for plane in ("Coronal", "Axial"):
                a, b = blocks[plane]
                volume[a:b] = volume[a:b, :, ::-1]           # coronal / axial: flip left-right
    except Exception:
        pass
    return volume


# ---------- v6: what is stored ----------
SLOTS_ALL = ["Sagittal", "Coronal", "Axial", "Sagittal2"]   # Sagittal2 = a second sagittal series WITHOUT fat suppression
SLOT_PLANE = {"Sagittal": "Sagittal", "Coronal": "Coronal", "Axial": "Axial", "Sagittal2": "Sagittal"}
# slices the model reads per series: "v5" = as before; "more" = closer to the real series length (median about 30)
PICKS_SETS = {"v5": {"Sagittal": 20, "Coronal": 16, "Axial": 12, "Sagittal2": 12},
              "more": {"Sagittal": 28, "Coronal": 24, "Axial": 24, "Sagittal2": 16}}


def pick_series_v6(series_df):
    """Same choice as v5 for the 3 main series (fluid-sensitive first), plus one plain sagittal series."""
    series_df = series_df.copy()
    for col, empty in (("Fluid_Sensitive", 0), ("Fat_Suppression", 0), ("Anatomical_Plane", ""), ("SeriesInstanceUID", "")):
        if col not in series_df.columns:
            series_df[col] = empty
    if "StudyInstanceUID" not in series_df.columns or series_df.empty:
        return pd.DataFrame(columns=["StudyInstanceUID"] + SLOTS_ALL)
    ranked = series_df.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
    rows = []
    for study, group in ranked.groupby("StudyInstanceUID"):
        row = {"StudyInstanceUID": str(study)}
        for plane in ("Sagittal", "Coronal", "Axial"):
            same = group[group["Anatomical_Plane"] == plane]
            row[plane] = str(same.iloc[0]["SeriesInstanceUID"]) if len(same) else ""
        plain = group[(group["Anatomical_Plane"] == "Sagittal") & (group["Fluid_Sensitive"] == 0)
                      & (group["Fat_Suppression"] == 0) & (group["SeriesInstanceUID"].astype(str) != row["Sagittal"])]
        row["Sagittal2"] = str(plain.iloc[0]["SeriesInstanceUID"]) if len(plain) else ""
        rows.append(row)
    return pd.DataFrame(rows)


def load_series_all(folder, prep):
    """EVERY slice inside the span (not only 20/16/12), same crop, size and window as v5. uint8 (N, size, size)."""
    size = prep["img_size"]
    paths = sorted_paths(folder)
    if not paths:
        return np.zeros((0, size, size), np.uint8)
    start = int(prep["span"][0] * len(paths))
    stop = max(int(prep["span"][1] * len(paths)), start + 1)
    idx = np.arange(start, stop)
    if len(idx) > prep["max_slices"]:                       # very long series: keep an even subset
        idx = np.unique(np.linspace(start, stop - 1, prep["max_slices"]).round().astype(int))
    slices, last_good = [], None
    for i in idx:
        image, mm = read_image(paths[i])
        if image is None:
            if last_good is not None:
                slices.append(last_good)
            continue
        image = cv2.resize(crop_to_mm(image, mm, prep["crop_mm"]), (size, size), interpolation=cv2.INTER_AREA)
        slices.append(image)
        last_good = image
    if not slices:
        return np.zeros((0, size, size), np.uint8)
    stack = np.stack(slices).astype(np.float32)
    low, high = np.percentile(stack, [1, 99])               # one window for the whole series (as v5)
    stack = (np.clip(stack, low, high) - low) / max(high - low, 1e-6)
    return (stack * 255).astype(np.uint8)


def load_study_v6(study_uid, row, root, prep, with_side=False):
    """{slot: uint8 (N, size, size)}. Left knees mirrored so every knee looks like a right knee (as v5)."""
    size = prep["img_size"]
    out = {}
    for slot in prep["slots"]:
        series_uid = row.get(slot, "")
        stack = np.zeros((0, size, size), np.uint8)
        if isinstance(series_uid, str) and series_uid:
            try:
                stack = load_series_all(os.path.join(root, study_uid, series_uid), prep)
            except Exception:
                pass
        out[slot] = stack
    side = None
    try:
        side = knee_side(study_uid, row, root)
        if side == "L":
            for slot, stack in out.items():
                out[slot] = np.ascontiguousarray(stack[::-1] if SLOT_PLANE[slot] == "Sagittal" else stack[:, :, ::-1])
    except Exception:
        pass
    return (out, side) if with_side else out


def encode_jpeg(image, quality):
    ok, buf = cv2.imencode(".jpg", image, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
    if not ok:
        raise IOError("JPEG encoding failed")
    return buf.tobytes()


def decode_jpeg(data):
    image = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise IOError("JPEG decoding failed")
    return image


def study_to_jpegs(study, prep):
    """{slot: uint8 stack} -> {slot: [jpeg bytes per slice]}"""
    return {slot: [encode_jpeg(s, prep["jpeg_quality"]) for s in study[slot]] for slot in prep["slots"]}


def psnr(a, b):
    mse = np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2)
    return 99.0 if mse == 0 else float(10 * np.log10(255.0 ** 2 / mse))

# ---------- v6: reading the store, picking slices, augmentation ----------
class StoreReader:
    """The JPEG image store (one or more parts made by notebook A6). Reads one study at a time with os.pread:
    no memory map, so RAM stays flat in every data worker (the v5 lesson)."""

    def __init__(self, folders):
        self.files, self.uids, self.entries = [], [], []
        self.prep = None
        for folder in folders:
            p = json.load(open(os.path.join(folder, "prep.json")))
            if self.prep is None:
                self.prep = {k: v for k, v in p.items() if k not in ("shard", "n_shards")}
            elif any(self.prep[k] != p[k] for k in self.prep):
                raise ValueError(f"image store parts were made with different settings: {folder}")
            for index_path in sorted(glob.glob(os.path.join(folder, "index_*.json"))):
                part = os.path.basename(index_path)[len("index_"):-len(".json")]
                data = json.load(open(index_path))
                fd = len(self.files)
                self.files.append(os.path.join(folder, f"store_{part}.bin"))
                for uid, entry in zip(data["uids"], data["studies"]):
                    self.uids.append(str(uid))
                    self.entries.append((fd, entry))
        dup = len(self.uids) - len(set(self.uids))
        if dup:
            raise ValueError(f"{dup} studies appear in more than one image store part (same part attached twice?)")
        self.fds = None

    def __len__(self):
        return len(self.uids)

    def _fd(self, k):
        if self.fds is None or self.fds[0] != os.getpid():       # each forked worker opens its own files
            self.fds = (os.getpid(), [os.open(f, os.O_RDONLY) for f in self.files])
        return self.fds[1][k]

    def __getitem__(self, i):
        """{slot: [jpeg bytes, ...]} for study row i."""
        fd_k, (offset, nbytes, slots) = self.entries[int(i)]
        fd = self._fd(fd_k)
        blob = os.pread(fd, nbytes, offset)
        if len(blob) != nbytes:
            raise IOError(f"image store: short read for study {self.uids[int(i)]}")
        try:
            os.posix_fadvise(fd, offset, nbytes, os.POSIX_FADV_DONTNEED)
        except (AttributeError, OSError):
            pass
        return {slot: [blob[a:a + n] for a, n in parts] for slot, parts in slots.items()}


def window_table_v6(slots, picks):
    """Every sliding 3-slice picture: (slot id, centre slice, position)."""
    rows = []
    for p, slot in enumerate(slots):
        for c in range(1, picks[slot] - 1):
            rows.append((p, c, c))
    return np.array(rows, dtype=np.int64)


def slice_positions(n_stored, n, training, aug, shift=0.0):
    """Where to read the n slices (fractional positions in the stored series).
    Prediction: evenly spread over the whole series (shift = part of a step, for the slice check).
    Training: shifted, squeezed and jittered a little, so every round reads other real slices
    (or a blend of two neighbours) instead of the same ones."""
    if n_stored <= 1:
        return np.zeros(n)
    base = np.linspace(0, n_stored - 1, n)
    stride = (n_stored - 1) / max(n - 1, 1)
    if not training:
        return np.clip(base + shift * stride, 0, n_stored - 1)
    pos = base.copy()
    if np.random.rand() < aug["p_span"]:
        L = n_stored - 1
        s = np.random.uniform(*aug["span"])                  # read a slightly shorter part of the series
        centre = np.random.uniform(s * L / 2, L - s * L / 2)
        pos = centre + (base - L / 2) * s
    pos = pos + np.random.uniform(-aug["slice_shift"], aug["slice_shift"]) * stride
    pos = pos + np.random.uniform(-aug["slice_jitter"], aug["slice_jitter"], size=n) * stride
    return np.sort(np.clip(pos, 0, n_stored - 1))


def crop_box(store_size, store_mm, view, training):
    """Which square of the stored picture the model sees. Prediction: the centre view_mm (as v5: 140 mm).
    Training: a random size between zoom_mm and a random place, so the knee is not always in the same spot."""
    mm_px = store_size / store_mm
    if training:
        side = min(np.random.uniform(*view["zoom_mm"]), store_mm) * mm_px
        y0 = np.random.uniform(0, store_size - side)
        x0 = np.random.uniform(0, store_size - side)
    else:
        side = min(view["view_mm"], store_mm) * mm_px
        y0 = x0 = (store_size - side) / 2
    y0, x0, side = int(round(y0)), int(round(x0)), max(int(round(side)), 8)
    return y0, x0, min(side, store_size - y0, store_size - x0)


def read_slices(jpegs, positions, needed, box, out_size):
    """float32 (n, out_size, out_size) in 0-1. Only the slices in `needed` are made (others stay 0).
    Each stored slice: decode -> cut the box -> resize. A fractional position is a blend of the two nearest
    stored slices: a short series gets in-between slices, not the same slice twice."""
    out = np.zeros((len(positions), out_size, out_size), np.float32)
    y0, x0, side = box
    decoded = {}

    def get(i):
        if i not in decoded:
            img = decode_jpeg(jpegs[i])[y0:y0 + side, x0:x0 + side]
            if img.shape[0] != out_size or img.shape[1] != out_size:
                img = cv2.resize(img, (out_size, out_size), interpolation=cv2.INTER_AREA if side > out_size else cv2.INTER_LINEAR)
            decoded[i] = img.astype(np.float32) / 255.0
        return decoded[i]

    for k in needed:
        x = positions[k]
        i0 = int(np.floor(x))
        i1 = min(i0 + 1, len(jpegs) - 1)
        t = x - i0
        out[k] = get(i0) if t < 1e-3 or i1 == i0 else (1 - t) * get(i0) + t * get(i1)
    return out


AUG_PRESETS = {
    # the new, stronger set. No left-right flip ever: medial and lateral must keep their meaning.
    "strong": dict(p_affine=0.8, rot=15, scale=(0.92, 1.08), shift=0.03, shear=5,
                   p_intensity=0.8, gamma=(0.7, 1.4), contrast=(0.85, 1.15), brightness=0.08,
                   p_bias=0.3, bias=0.25, p_lowres=0.15, lowres=(0.6, 0.9), p_blur=0.15, blur=(0.4, 1.0),
                   p_noise=0.3, noise=(0.0, 0.04), p_erase=0.2, erase=(0.08, 0.2), p_drop_slot=0.05,
                   slice_shift=0.5, slice_jitter=0.15, p_span=0.5, span=(0.85, 1.0)),
    # the old v5 set on the new store, for a fair comparison (only the fresh slices are new)
    "v5": dict(p_affine=0.7, rot=10, scale=(0.9, 1.1), shift=0.05, shear=0,
               p_intensity=1.0, gamma=(0.85, 1.15), contrast=(0.9, 1.1), brightness=0.0,
               p_bias=0.0, bias=0.0, p_lowres=0.0, lowres=(1, 1), p_blur=0.0, blur=(0, 0),
               p_noise=0.0, noise=(0, 0), p_erase=0.0, erase=(0, 0), p_drop_slot=0.0,
               slice_shift=0.5, slice_jitter=0.15, p_span=0.5, span=(0.85, 1.0)),
}


def affine_matrix(size, aug):
    """Rotation, zoom, shear and shift around the picture centre."""
    a = np.deg2rad(np.random.uniform(-aug["rot"], aug["rot"]))
    s = np.random.uniform(*aug["scale"])
    sh = np.tan(np.deg2rad(np.random.uniform(-aug["shear"], aug["shear"])))
    A = s * np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]]) @ np.array([[1, sh], [0, 1]])
    c = np.array([size / 2, size / 2])
    t = c - A @ c + np.random.uniform(-aug["shift"], aug["shift"], size=2) * size
    return np.hstack([A, t[:, None]]).astype(np.float32)


def bias_field(size, strength):
    """A smooth brighter/darker shading, like an MRI coil gives."""
    f = cv2.resize(np.random.randn(4, 4).astype(np.float32), (size, size), interpolation=cv2.INTER_CUBIC)
    return np.exp(strength * f / max(np.abs(f).max(), 1e-6))


def augment_slot(stack, needed, aug):
    """Same change for every slice of one series, so the 3-slice pictures stay lined up."""
    size = stack.shape[-1]
    idx = list(needed)
    if np.random.rand() < aug["p_affine"]:
        m = affine_matrix(size, aug)
        for k in idx:
            stack[k] = cv2.warpAffine(stack[k], m, (size, size), flags=cv2.INTER_LINEAR, borderValue=0)
    if np.random.rand() < aug["p_intensity"]:
        g = np.random.uniform(*aug["gamma"])
        c = np.random.uniform(*aug["contrast"])
        b = np.random.uniform(-aug["brightness"], aug["brightness"]) if aug["brightness"] else 0.0
        stack[idx] = np.clip(np.clip(stack[idx], 0, 1) ** g * c + b, 0, 1)
    if np.random.rand() < aug["p_bias"]:
        stack[idx] = np.clip(stack[idx] * bias_field(size, aug["bias"])[None], 0, 1)
    if np.random.rand() < aug["p_lowres"]:
        small = max(int(size * np.random.uniform(*aug["lowres"])), 16)
        for k in idx:
            stack[k] = cv2.resize(cv2.resize(stack[k], (small, small), interpolation=cv2.INTER_AREA),
                                  (size, size), interpolation=cv2.INTER_LINEAR)
    if np.random.rand() < aug["p_blur"]:
        sigma = np.random.uniform(*aug["blur"])
        for k in idx:
            stack[k] = cv2.GaussianBlur(stack[k], (0, 0), sigma)
    return stack


def augment_pictures(x, aug):
    """Per 3-slice picture: a little noise, sometimes one small erased box."""
    n, _, h, w = x.shape
    for i in range(n):
        if np.random.rand() < aug["p_noise"]:
            x[i] += np.random.normal(0, np.random.uniform(*aug["noise"]), size=x[i].shape).astype(np.float32)
        if np.random.rand() < aug["p_erase"]:
            frac = np.random.uniform(*aug["erase"])
            bh, bw = int(h * np.sqrt(frac)), int(w * np.sqrt(frac))
            y0, x0 = np.random.randint(0, h - bh + 1), np.random.randint(0, w - bw + 1)
            x[i, :, y0:y0 + bh, x0:x0 + bw] = 0
    return np.clip(x, 0, 1)


def make_windows_v6(study, slots, table, chosen, view, store, training, aug, shift=0.0):
    """study: {slot: [jpeg, ...]}; view: picks per series, model picture size, field of view;
    store: the store's prep (picture size and mm). Returns (len(chosen), 3, H, W) floats, slot ids, positions, valid."""
    rows = table[chosen]
    size = view["size"]
    x = np.zeros((len(rows), 3, size, size), np.float32)
    for p, slot in enumerate(slots):
        mine = np.where(rows[:, 0] == p)[0]
        data = study.get(slot, [])
        if not len(mine) or len(data) == 0:
            continue
        if training and np.random.rand() < aug["p_drop_slot"]:
            continue                                         # sometimes a whole series is missing: learn to live with it
        positions = slice_positions(len(data), view["picks"][slot], training, aug, shift)
        needed = sorted({c + d for c in rows[mine, 1] for d in (-1, 0, 1)})
        box = crop_box(store["img_size"], store["crop_mm"], view, training)
        stack = read_slices(data, positions, needed, box, size)
        if training:
            stack = augment_slot(stack, needed, aug)
        for r in mine:
            c = rows[r, 1]
            x[r] = stack[c - 1:c + 2]
    if training:
        x = augment_pictures(x, aug)
    valid = x.reshape(len(rows), -1).max(axis=1) > 0
    return x, rows[:, 0], rows[:, 2], valid


# ---------- v7: anatomical mirror (training augmentation and test-time view) ----------
def mirror_study(study):
    """The same knee seen as its mirror image (a right knee looks like a left one).
    Sagittal series: read backwards. Coronal / axial: flipped later, picture by picture (flip_windows)."""
    return {slot: (list(data)[::-1] if SLOT_PLANE.get(slot) == "Sagittal" else data) for slot, data in study.items()}


def flip_windows(x, plane_ids, slots):
    """Flip left-right every picture of a coronal or axial series (after mirror_study)."""
    flip = np.array([SLOT_PLANE.get(slots[p]) != "Sagittal" for p in plane_ids])
    if flip.any():
        x[flip] = x[flip][..., ::-1]
    return np.ascontiguousarray(x)


In [4]:
# ---------- decoder test: every transfer syntax in this data must decode ----------
ROOT = os.path.join(COMP, "train_series")
assert os.path.isdir(ROOT), f"train_series/ not found in {COMP}: {sorted(os.listdir(COMP))}"
rng = np.random.default_rng(PART)
studies = sorted(os.listdir(ROOT))
sample = []
for s in rng.choice(studies, size=min(150, len(studies)), replace=False):
    for se in sorted(os.listdir(os.path.join(ROOT, s))):
        files = sorted(glob.glob(os.path.join(ROOT, s, se, "*.dcm")))
        if files:
            sample.append(files[len(files) // 2])
by_syntax = {}
for path in sample:
    try:
        ts = str(pydicom.dcmread(path, stop_before_pixels=True).file_meta.TransferSyntaxUID.name)
    except Exception:
        ts = "unreadable header"
    ok = read_image(path)[0] is not None
    good, total = by_syntax.get(ts, (0, 0))
    by_syntax[ts] = (good + ok, total + 1)
for ts, (good, total) in sorted(by_syntax.items()):
    log(f"  {ts:70s} decoded {good}/{total}")
bad = {ts: v for ts, v in by_syntax.items() if v[0] < v[1]}
assert not bad, (f"some DICOMs cannot be decoded: {bad}. Turn Internet ON (Settings) so the decoders install, "
                 "then run again. Otherwise those slices would be stored black.")
log("decoder test passed: every transfer syntax decodes")


[   0.7 min]   Explicit VR Little Endian                                              decoded 839/839
[   0.7 min] decoder test passed: every transfer syntax decodes


In [5]:
# ---------- which studies, where their DICOM files are ----------
series = pd.read_csv(f"{COMP}/train_series.csv")
for col in ("StudyInstanceUID", "SeriesInstanceUID"):
    series[col] = series[col].astype(str)
chosen = pick_series_v6(series).set_index("StudyInstanceUID")
train = pd.read_csv(f"{COMP}/train.csv")
uids = sorted(set(train["StudyInstanceUID"].astype(str)))
my_uids = [u for k, u in enumerate(uids) if k % N_PARTS == PART]


ROOT = os.path.join(COMP, "train_series")       # train_series/<study>/<series>/<instance>.dcm (data page)
PREP = {"version": "v7-jpeg-384-160", "img_size": IMG_SIZE, "crop_mm": CROP_MM, "span": SPAN,
        "max_slices": MAX_SLICES, "jpeg_quality": JPEG_QUALITY, "slots": SLOTS_ALL}
log(f"DICOM folder: {ROOT} | studies: {len(uids)} | this part ({PART} of {N_PARTS}): {len(my_uids)}")
log(f"settings: {PREP}")
missing = [u for u in my_uids if u not in chosen.index]
if missing:
    log(f"warning: {len(missing)} studies have no row in train_series.csv; they are stored with empty series")


_read_image_plain = read_image
READ_FAILS = [0]


def read_image(path):
    """Same as before, but counts the slices that could not be read (they would be skipped silently)."""
    out = _read_image_plain(path)
    if out[0] is None:
        READ_FAILS[0] += 1
    return out


def build_one(uid):
    """One study -> ({slot: [jpeg bytes per slice]}, error, unreadable slices). Never raises."""
    READ_FAILS[0] = 0
    try:
        row = chosen.loc[uid].to_dict() if uid in chosen.index else {}
        study = load_study_v6(uid, row, ROOT, PREP)
        return uid, study_to_jpegs(study, PREP), None, READ_FAILS[0]
    except Exception as e:
        return uid, {slot: [] for slot in PREP["slots"]}, f"{type(e).__name__}: {e}", READ_FAILS[0]


def save_index(entries, done_uids):
    """Written under a temporary name first, so a stop never leaves a broken index."""
    tmp = f"{OUT}/index_{PART}.json.tmp"
    json.dump({"uids": done_uids, "studies": entries}, open(tmp, "w"))
    os.replace(tmp, f"{OUT}/index_{PART}.json")


cv2.setNumThreads(1)
json.dump(dict(PREP, shard=PART, n_shards=N_PARTS), open(f"{OUT}/prep.json", "w"), indent=1)
bin_path = f"{OUT}/store_{PART}.bin"
entries, done_uids, errors, offset, unreadable = [], [], [], 0, 0
with open(bin_path, "wb") as fout, Pool(WORKERS) as pool:
    for k, (uid, jpegs, err, fails) in enumerate(pool.imap(build_one, my_uids, chunksize=2)):
        unreadable += fails
        start, slots = offset, {}
        for slot in PREP["slots"]:
            parts = []
            for data in jpegs.get(slot, []):
                fout.write(data)
                parts.append([offset - start, len(data)])
                offset += len(data)
            slots[slot] = parts
        entries.append([start, offset - start, slots])
        done_uids.append(uid)
        if err:
            errors.append((uid, err))
            log(f"  warning: study {uid} stored empty ({err})")
        if (k + 1) % 100 == 0 or k + 1 == len(my_uids):
            fout.flush()
            save_index(entries, done_uids)
            gb = offset / 1e9
            per = (time.time() - T0) / (k + 1)
            log(f"  {k + 1}/{len(my_uids)} studies | {gb:.2f} GB | {per:.2f} s per study | "
                f"about {per * (len(my_uids) - k - 1) / 60:.0f} min left")
            if gb / (k + 1) * len(my_uids) > MAX_PART_GB and k + 1 < len(my_uids):
                log(f"STOP: this part would pass {MAX_PART_GB} GB. Use N_PARTS = {N_PARTS * 2} and run again.")
                break
save_index(entries, done_uids)
log(f"part {PART} done: {len(done_uids)} studies, {offset / 1e9:.2f} GB, {len(errors)} stored empty, "
    f"{unreadable} DICOM slices could not be read")


[   0.8 min] DICOM folder: /kaggle/input/competitions/rsna-knee-abnormality-detection/train_series | studies: 4407 | this part (3 of 4): 1101
[   0.8 min] settings: {'version': 'v7-jpeg-384-160', 'img_size': 384, 'crop_mm': 160.0, 'span': [0.06, 0.94], 'max_slices': 64, 'jpeg_quality': 95, 'slots': ['Sagittal', 'Coronal', 'Axial', 'Sagittal2']}
[   2.6 min]   100/1101 studies | 0.46 GB | 1.54 s per study | about 26 min left
[   4.0 min]   200/1101 studies | 0.91 GB | 1.20 s per study | about 18 min left
[   5.6 min]   300/1101 studies | 1.36 GB | 1.12 s per study | about 15 min left
[   7.3 min]   400/1101 studies | 1.82 GB | 1.10 s per study | about 13 min left
[   8.9 min]   500/1101 studies | 2.25 GB | 1.07 s per study | about 11 min left
[  10.5 min]   600/1101 studies | 2.71 GB | 1.05 s per study | about 9 min left
[  12.1 min]   700/1101 studies | 3.17 GB | 1.03 s per study | about 7 min left
[  13.9 min]   800/1101 studies | 3.64 GB | 1.05 s per study | about 5 min left
[  15.9 

In [6]:
# ---------- check the part before saving it as a dataset ----------
reader = StoreReader([OUT])
assert reader.prep == {k: v for k, v in PREP.items()}, "prep.json differs from the settings"
assert len(reader) == len(my_uids), f"only {len(reader)} of {len(my_uids)} studies are in this part"
assert set(reader.uids) == set(my_uids)
empty_slots = {s: 0 for s in PREP["slots"]}
for i in range(len(reader)):
    study = reader[i]
    for s in PREP["slots"]:
        empty_slots[s] += len(study[s]) == 0
log(f"every study readable; series missing per slot (Sagittal2 is often absent): {empty_slots}")
# decode every slice of a few studies and compare with a fresh read (JPEG must be close to the original)
rng = np.random.default_rng(0)
for i in rng.choice(len(reader), size=min(3, len(reader)), replace=False):
    uid = reader.uids[i]
    study = reader[i]
    fresh = load_study_v6(uid, chosen.loc[uid].to_dict() if uid in chosen.index else {}, ROOT, PREP)
    for s in PREP["slots"]:
        dec = [decode_jpeg(b) for b in study[s]]
        assert len(dec) == len(fresh[s]), f"{uid} {s}: {len(dec)} slices stored, {len(fresh[s])} read"
        assert all(d.shape == (IMG_SIZE, IMG_SIZE) for d in dec)
        if dec:
            q = min(psnr(d, f) for d, f in zip(dec, fresh[s]))
            assert q > 30, f"{uid} {s}: JPEG quality too low (PSNR {q:.1f})"
    log(f"  study {uid}: slices per slot {[len(study[s]) for s in PREP['slots']]} decode and match the DICOM read")
n_slices = sum(len(e[2][s]) for e in entries for s in e[2])
log(f"slices stored: {n_slices}; DICOM slices that could not be read: {unreadable} ({unreadable / max(n_slices, 1):.3%})")
assert unreadable <= 0.001 * max(n_slices, 1), "too many unreadable DICOM slices: check the decoder test above"
# optional: compare with the old v6 store (attach one old part): the same studies must have at least as many slices
old_dirs = [os.path.dirname(p) for p in [q for d in range(1, 5) for q in glob.glob("/kaggle/input/" + "*/" * d + "prep.json")]
            if json.load(open(p)).get("version", "").startswith("v6") and glob.glob(os.path.join(os.path.dirname(p), "index_*.json"))]
if old_dirs:
    old = StoreReader(old_dirs[:1])
    common = [u for u in old.uids if u in set(reader.uids)]
    old_row, new_row = {u: i for i, u in enumerate(old.uids)}, {u: i for i, u in enumerate(reader.uids)}
    fewer = more = 0
    for u in common[:300]:
        a, b = old[old_row[u]], reader[new_row[u]]
        for s in PREP["slots"]:
            fewer += len(b[s]) < len(a[s])
            more += len(b[s]) > len(a[s])
    log(f"compared with the old v6 store on {min(len(common), 300)} studies: series with MORE slices now {more} "
        f"(slices the old store could not read), with FEWER {fewer}")
    assert fewer == 0, "the new store has fewer slices than the old one for some series: do not use it"
if errors:
    log(f"{len(errors)} studies were stored empty: {errors[:5]}")
    assert len(errors) < 0.01 * len(my_uids), "more than 1% of studies failed: check the warnings above"
log(f"CHECK PASSED. Output -> New Dataset -> name it knee-store-v7-{PART}")


[  19.4 min] every study readable; series missing per slot (Sagittal2 is often absent): {'Sagittal': 0, 'Coronal': 0, 'Axial': 0, 'Sagittal2': 38}
[  19.5 min]   study 1.2.826.0.1.3680043.8.498.56991590802231436015843619633497607680: slices per slot [22, 22, 19, 22] decode and match the DICOM read
[  19.5 min]   study 1.2.826.0.1.3680043.8.498.41440206520769979754790425253631809212: slices per slot [30, 27, 29, 30] decode and match the DICOM read
[  19.5 min]   study 1.2.826.0.1.3680043.8.498.81892104897856100357169774214718573167: slices per slot [30, 27, 27, 30] decode and match the DICOM read
[  19.5 min] slices stored: 120054; DICOM slices that could not be read: 0 (0.000%)
[  19.7 min] CHECK PASSED. Output -> New Dataset -> name it knee-store-v7-3
